# 09 · Operasional & Arah Riset — Bab 10

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 10: monitoring drift (grafik kendali), ensembel multi-seed untuk ketidakpastian, regresi kuantil, dan SHAP untuk interpretasi.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

## 2. Data Sederhana (regresi)

Gunakan data sintetik agar latihan singkat; prinsip identik untuk Bab 8/9.

In [ ]:
n = 300
x = np.linspace(-3,3,n)[:,None]
y = np.sin(2*x).ravel() + 0.3*np.random.randn(n)
ntr = int(n*0.7)
xtr, ytr = x[:ntr], y[:ntr]
xte, yte = x[ntr:], y[ntr:]
print("train", xtr.shape, "test", xte.shape)

## 3. Ensembel Multi-Seed (Kode 10.1)

Latih model yang sama dengan seed berbeda; rata-rata & std sebagai ketidakpastian.

In [ ]:
def make_model():
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation="relu", input_shape=(1,)),
        tf.keras.layers.Dense(1)])
    m.compile(optimizer="adam", loss="mse")
    return m

preds = []
for seed in [1, 2, 3, 5]:
    tf.keras.utils.set_random_seed(seed)
    m = make_model()
    m.fit(xtr, ytr, epochs=200, verbose=0)
    preds.append(m.predict(xte, verbose=0).ravel())

bay = np.stack(preds)
p_mean, p_std = bay.mean(0), bay.std(0)
print("prediksi rata-rata[:5]:", p_mean[:5].round(3))
print("std (ketidakpastian)[:5]:", p_std[:5].round(3))

In [ ]:
ix = np.argsort(xte.ravel())
plt.figure(figsize=(7,3.6))
plt.fill_between(xte.ravel()[ix], p_mean[ix]-2*p_std[ix], p_mean[ix]+2*p_std[ix],
                 alpha=0.25, color="#4a90e2", label="±2σ ensambel")
plt.plot(xte.ravel()[ix], p_mean[ix], color="#2c5f8a", lw=1.5, label="rata-rata")
plt.plot(xte.ravel()[ix], yte[ix], "o", ms=2.5, alpha=0.5, label="aktual test")
plt.xlabel("x"); plt.ylabel("y"); plt.legend()
plt.title("Ensembel multi-seed: rata-rata + ketidakpastian")
plt.tight_layout(); plt.show()

## 4. Regresi Kuantil (Kode 10.2)

In [ ]:
def loss_kuantil(q):
    def _loss(yt, yp):
        err = yt - yp
        return tf.reduce_mean(tf.maximum(q*err, (q-1)*err))
    return _loss

mq = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation="relu", input_shape=(1,)),
    tf.keras.layers.Dense(2)])
mq.compile(optimizer="adam", loss=[loss_kuantil(0.10), loss_kuantil(0.90)])
mq.fit(xtr, np.stack([ytr, ytr], -1), epochs=200, verbose=0)
out = mq.predict(xte, verbose=0)
qlo, qhi = out[:,0], out[:,1]
coverage = float(np.mean((yte >= qlo) & (yte <= qhi)))
print("Cakupan interval 10-90%:", round(coverage, 3))
print("Idealnya mendekati 0.80 (kalibrasi sempurna).")

## 5. Grafik Kendali MAE Mingguan (Gambar 10.1)

In [ ]:
np.random.seed(7)
mae26 = 0.10 + 0.02*np.random.randn(52)
mae26[38] = 0.16
mean = mae26[:25].mean(); sd = mae26[:25].std()

plt.figure(figsize=(7.5,3.4))
plt.plot(np.arange(52), mae26, "o-", ms=4, lw=1.2, color="#4a90e2")
plt.axhline(mean, ls="--", color="gray", lw=1, label="rata-rata")
plt.axhline(mean+2*sd, ls=":", color="#e74c3c", lw=1, label="+2σ")
plt.axhline(mean-2*sd, ls=":", color="#e74c3c", lw=1)
plt.scatter([38],[mae26[38]], color="#e74c3c", zorder=5, label="keluar batas")
plt.xlabel("Minggu"); plt.ylabel("MAE"); plt.legend(fontsize=8)
plt.title("Grafik kendali MAE mingguan")
plt.tight_layout(); plt.show()

## 6. Catatan SHAP (Kode 10.3)

SHAP butuh paket `shap` — instal bila belum: `pip install shap`. Contoh ringkas pada model Keras:

```python
import shap
explainer = shap.Explainer(m.predict, xtr[:50])
vals = explainer(xte[:50])
shap.plots.beeswarm(vals)
```

Jalankan di sel terpisah (butuh waktu & paket tambahan).

## 7. Latihan Mini

1. Ubah seed ensambel menjadi {1,2,3,4,5} — apakah rata-rata stabil? std mengecil?
2. Terapkan regresi kuantil pada model Bab 8/9 dengan fitur asli; ukur cakupan interval.
3. Palsukan drift pada data uji (geser target) dan deteksi dengan grafik kendali.
4. (Proyek) Susun rencana operasional satu halaman untuk salah satu studi kasus.